In [41]:
import numpy as np
import toughio
import re

from pathlib import Path


def _parse_chemical_section_names(chemical_path, section_header="# Aqueous complexes"):
    chemical_path = Path(chemical_path)
    if not chemical_path.exists():
        return []

    names = []
    in_section = False
    with chemical_path.open("r", encoding="utf-8", errors="ignore") as f:
        for raw in f:
            line = raw.strip()
            if line.startswith(section_header):
                in_section = True
                continue
            if not in_section:
                continue
            if line == "*":
                break
            if not line or line.startswith("#"):
                continue
            names.append(line.split()[0])
    return names


def _patch_solute_header_block(text):
    pattern = (
        r"#MAXITPTR\s+TOLTR\s+MAXITPCH\s+TOLCH\s+TOLMB\s+TOLDC\s+TOLDR\s*\n"
        r"\s*40\s+0\.0001\s+1000\s+0\.001\s+30\.0\s+0\.001\s+0\.0\s*"
    )
    replacement = (
        "# MAXITPTR  TOLTR    MAXITPCH  TOLCH    MAXITPAD  TOLAD    TOLDC    TOLDR\n"
        "40 0.100E-03  1000 1.000E-03   30 0.100E-02 0.000E-03 0.000E-03     ! ....TOLDC,TOLDR\n"
    )
    return re.sub(pattern, replacement, text, count=1)


def _set_nwaq_minus_one(lines):
    for i, line in enumerate(lines):
        if line.startswith("# NWTI"):
            j = i + 1
            parts = lines[j].split()
            if len(parts) < 10:
                raise ValueError("Failed to parse the NWTI/NWCOM/NWMIN/NWAQ control line in solute.inp.")
            parts[4] = "-1"
            lines[j] = " ".join(parts)
            return
    raise ValueError("Could not find the NWTI/NWCOM/NWMIN/NWAQ control line in solute.inp.")


def _replace_individual_aqueous_species_block(lines, replacement_items):
    start = None
    end = None
    for i, line in enumerate(lines):
        if line.strip() == "# Individual aqueous species":
            start = i + 1
            continue
        if start is not None and line.strip() == "# Adsorption species":
            end = i
            break

    if start is None or end is None:
        raise ValueError("Could not locate the 'Individual aqueous species' block in solute.inp.")

    lines[start:end] = replacement_items + [""]
    return lines


def _format_int_record(indices):
    return " ".join(str(i) for i in indices)


def _postprocess_solute_input(solute_path, aqueous_species_indices, chemical_path="chemical.inp"):
    solute_path = Path(solute_path)
    text = solute_path.read_text(encoding="utf-8", errors="ignore")
    text = _patch_solute_header_block(text)

    int_line = _format_int_record(aqueous_species_indices)
    need_name_mode = len(int_line) > 200

    lines = text.splitlines()

    if need_name_mode:
        aq_names = _parse_chemical_section_names(chemical_path, "# Aqueous complexes")
        if not aq_names:
            raise FileNotFoundError(
                f"NWAQ automatic conversion was triggered because the integer record is too long, "
                f"but no readable chemical.inp was found at: {chemical_path}"
            )

        bad_indices = [i for i in aqueous_species_indices if i < 1 or i > len(aq_names)]
        if bad_indices:
            raise ValueError(
                f"NWAQ automatic conversion was triggered, but some aqueous species indices are out of range "
                f"for chemical.inp: {bad_indices[:10]}"
            )

        selected_names = []
        seen = set()
        for idx in aqueous_species_indices:
            name = aq_names[idx - 1]
            if name not in seen:
                selected_names.append(name)
                seen.add(name)

        _set_nwaq_minus_one(lines)
        _replace_individual_aqueous_species_block(lines, selected_names)
        text = "\n".join(lines) + "\n"
    else:
        text = "\n".join(lines) + "\n"

    solute_path.write_text(text, encoding="utf-8")


def finalize_solute_input(parameters, solute_path="solute.inp", chemical_path="chemical.inp"):
    toughio.write_input(solute_path, parameters, "toughreact-solute")
    aqueous_species_indices = parameters.get("output", {}).get("aqueous_species", [])
    _postprocess_solute_input(solute_path, aqueous_species_indices, chemical_path=chemical_path)


In [44]:
finalize_solute_input(parameters, solute_path="solute.inp", chemical_path="chemical.inp")
